# 📈 City Guardian — Independent Model Evaluation & Failure Mode Analysis
### Production Benchmarking Across Adverse Conditions

This notebook computes rigorous test-set metrics:
1. **mAP@50 & mAP@50-95**: Primary object detection localization precision.
2. **Latency & Throughput**: Preprocess, inference, and postprocess timing in ms and FPS.
3. **Per-Class Breakdown**: Pinpoints lagging categories.
4. **Visual Prediction Gallery**: Overlays predicted bounding boxes and confidences for visual quality verification.

In [ ]:
import sys
from pathlib import Path

# Set project root
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.detection.evaluator import ModelEvaluator
print('Evaluator module imported successfully!')

## 1. Benchmark Trained Domain Checkpoint

In [ ]:
DOMAIN = 'solid_waste'
WEIGHTS = project_root / 'models' / 'trained' / f'{DOMAIN}_yolov8.pt'
DATA_YAML = project_root / 'models' / 'configs' / f'{DOMAIN}.yaml'

if WEIGHTS.exists():
    evaluator = ModelEvaluator(str(WEIGHTS))
    summary = evaluator.evaluate(
        data_yaml=str(DATA_YAML),
        split='test',
        conf_threshold=0.25,
        iou_threshold=0.50
    )
else:
    print(f'Trained checkpoint not found at: {WEIGHTS}. Train model in Notebook 02 first.')

## 2. Generate Visual Prediction Gallery on Test Photos

In [ ]:
test_images_dir = project_root / 'data' / 'processed' / DOMAIN / 'images' / 'test'
gallery_dir = project_root / 'reports' / 'evaluation' / 'gallery'

if test_images_dir.exists() and WEIGHTS.exists():
    gallery = evaluator.generate_visual_gallery(
        test_images_dir=test_images_dir,
        output_dir=gallery_dir,
        conf_threshold=0.30
    )
    print(f'Generated {len(gallery)} gallery images.')
else:
    print('Test images or weights not found.')